# Türkçe Verifiable QA Pipeline — Colab Runner

Bu defter [pqa](https://github.com/cxrbon16/pqa) reposundaki pipeline'ı Colab'da uçtan uca çalıştırır.

**Önce:** `Çalışma zamanı > Çalışma zamanı türünü değiştir` menüsünden **G4** (NVIDIA RTX PRO 6000,
~96GB VRAM) seç. Bu defterdeki model seçimi G4'ün bol VRAM'ını kullanacak şekilde yapıldı —
T4/L4 gibi daha küçük bir GPU'daysan aşağıdaki "Daha küçük GPU" notuna bak. Sonra hücreleri sırayla
çalıştır.

In [ ]:
!nvidia-smi

## 1. Repoyu klonla ve bağımlılıkları kur

Colab'ın kendi Python ortamı zaten oturuma özel (izole) olduğu için ayrı bir venv gerekmiyor.

In [ ]:
!git clone https://github.com/cxrbon16/pqa.git
%cd pqa
!pip install -q -r requirements.txt

## 2. Ollama'yı kur, başlat, üretici modeli indir

Validator/çözücü modeller şimdilik devre dışı (`config.yaml` → `solving.solvers: []`) — sadece
soru üretici çalışıyor:

| Rol | Model | Neden |
|---|---|---|
| Üretici (soru üretimi) | `gemma4:31b-it-bf16` | Gemma 4'ün en güçlü dense boyutu, 16-bit — validator'lar kapalı olduğu için tek model çalışıyor, G4'ün ~96GB VRAM'ı bf16'yı rahatça kaldırıyor, en yüksek kalite bunda |

Tag [ollama.com/library](https://ollama.com/library) üzerinden doğrulandı — Gemma 4'ün `-it-`
(instruct) etiketli varyantları quantization sonekini `31b-it-bf16` gibi ortada taşıyor, salt
`31b-bf16` **geçersizdir**.

**Toplam indirme ~63GB** — bant genişliğine göre 15-30 dakika sürebilir. Disk kotanı önce kontrol
et. Çözücüleri daha sonra geri açmak istersen `config.yaml`'daki yorumlu blokları uncomment et
(bkz. ENTRY.md).

In [ ]:
!df -h /

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd lshw pciutils  # zstd: ollama install.sh çıkarma için; lshw/pciutils: GPU tespit uyarısını gidermek için
!curl -fsSL https://ollama.com/install.sh | sh
!which ollama || echo "UYARI: ollama PATH'te bulunamadı, yukarıdaki kurulum çıktısını kontrol et"

In [ ]:
import os, shutil, subprocess, time, urllib.request

# Validator'lar kapalıyken tek model (üretici) yükleniyor, bu ayarın şu an pratik etkisi yok;
# çözücüleri config.yaml'da geri açarsan aynı anda birden fazla model belleğe alınabilsin diye
# duruyor (G4'ün ~96GB VRAM'i buna izin verir).
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "3"

ollama_bin = shutil.which("ollama") or next(
    (p for p in ("/usr/local/bin/ollama", "/usr/bin/ollama") if os.path.exists(p)), None
)
if ollama_bin is None:
    raise RuntimeError(
        "ollama binary bulunamadı. Bir önceki hücredeki kurulumu tekrar çalıştır "
        "(curl -fsSL https://ollama.com/install.sh | sh) ve çıktısında hata olup olmadığına bak."
    )

log_path = "/content/ollama_serve.log"
log_file = open(log_path, "w")
ollama_proc = subprocess.Popen(
    [ollama_bin, "serve"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    # Colab'da bir hücreyi durdurmak (SIGINT) aynı process group'taki arka plan sürecini de
    # öldürebilir — start_new_session bunu ayrı bir session'a alıp bu sorundan korur. Sunucunun
    # "could not connect to ollama server" hatasıyla sessizce ölmesinin en sık nedeni budur.
    start_new_session=True,
)

for _ in range(30):
    if ollama_proc.poll() is not None:
        raise RuntimeError(
            f"ollama serve başlatılamadı (çıkış kodu {ollama_proc.returncode}). Log:\n"
            + open(log_path).read()
        )
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama serve 30 saniyede ayağa kalkmadı. Log:\n" + open(log_path).read())

print("ollama serve pid:", ollama_proc.pid, "| binary:", ollama_bin, "| log:", log_path)

# Ollama'nın GPU'yu gerçekten bulup bulmadığını logdan doğrula (kurulumdaki lspci/lshw
# uyarısı kozmetik olabilir — asıl kanıt bu satırlar).
print("\n--- ollama serve log (GPU ile ilgili satırlar) ---")
with open(log_path) as f:
    for line in f:
        if any(k in line.lower() for k in ("gpu", "cuda", "nvidia", "vram")):
            print(line.strip())

In [ ]:
!ollama pull gemma4:31b-it-bf16

In [ ]:
import json, subprocess

# Kesin GPU kontrolü: üretici modeli bir kez çalıştır, sonra /api/ps'teki size_vram'e bak.
# size_vram > 0 ve size_vram ~= size ise model (neredeyse) tamamen GPU'da — gerçek kanıt bu,
# kurulumdaki lspci/lshw uyarısı değil.
subprocess.run(["ollama", "run", "gemma4:31b-it-bf16", "merhaba"], capture_output=True, text=True, timeout=300)
ps = json.loads(subprocess.run(["curl", "-s", "http://localhost:11434/api/ps"], capture_output=True, text=True).stdout)
for m in ps.get("models", []):
    size_gb, vram_gb = m["size"] / 1e9, m["size_vram"] / 1e9
    print(f"{m['name']}: size={size_gb:.1f}GB, size_vram={vram_gb:.1f}GB -> "
          f"{'GPU üzerinde ✓' if vram_gb / size_gb > 0.9 else 'KISMEN/TAMAMEN CPU! sorunu araştır'}")

In [ ]:
# Sunucu ayakta mı ve modeller indi mi kontrol et
!curl -s http://localhost:11434/api/tags | python3 -m json.tool

**Daha küçük GPU (T4/L4) kullanıyorsan:** `gemma4:31b-it-bf16` (~63GB) T4/L4'te (16-24GB) sığmaz.
`config.yaml`'da üretici modeli daha küçük bir tag ile değiştir, örn.:

```yaml
generation:
  model:
    model: gemma4:e4b-it-q8_0   # ~10GB, T4'te rahat çalışır
```

ve `ollama pull gemma4:e4b-it-q8_0` ile indir.

## 3. Duman testi (smoke test): küçük bir örnekle uçtan uca çalıştır

`--limit N` her aşamanın girdisini ilk N kayıtla sınırlar. Önce ucuz aşamaları (fetch/passages/
generate/filter) çalıştırıp çıktıyı gözle kontrol et. **Solve + band (çözücülerin soruyu gerçekten
çözüp çözemediğini doğrulayan, yani "verify" adımı) ayrı bir hücrede** — özellikle `llama4:scout`
gibi büyük modellerin GPU'ya her seferinde yeniden yüklenmesi yüzünden en yavaş kısım burası;
istersen bu adıma geçmeden önce üsttekileri birkaç kez deneyip iyileştirebilirsin.

In [ ]:
!python -m vqa fetch --limit 5
!python -m vqa passages
!python -m vqa generate
!python -m vqa filter

In [ ]:
import json

for line in open("data/04_filtered.jsonl", encoding="utf-8"):
    rec = json.loads(line)
    print(rec["question"], "->", rec["answer"])

### 3b. Solve + Band (opsiyonel — validator'lar kapalı)

`config.yaml` → `solving.solvers: []` olduğu için bu hücre artık hiçbir modeli çağırmaz: her
kaydı olduğu gibi `difficulty="unverified"` etiketiyle geçirir (çökmez, sadece işaretler).
Çözülebilirlik doğrulamasını daha sonra açmak istersen `config.yaml`'daki yorumlu çözücü
bloklarını uncomment edip bu hücreyi tekrar çalıştır — o zaman gerçek solve/band mantığı devreye
girer.

In [ ]:
!python -m vqa solve
!python -m vqa band

In [ ]:
import json

for line in open("data/06_final.jsonl", encoding="utf-8"):
    rec = json.loads(line)
    print(f"[{rec['difficulty']:>6} | solve_rate={rec['solve_rate']}] {rec['question']} -> {rec['answer']}")

## 4. Google Drive'a bağla (oturum kesilirse veri kaybolmasın)

Colab oturumları zaman aşımına uğrayabilir/kopabilir; her diskteki her şey oturumla birlikte
silinir. `data_dir`'i Drive'a taşımak, uzun bir çalıştırmayı kaldığı yerden sürdürmeni sağlar —
her aşama kendi JSONL'ini yazdığı için pipeline zaten yeniden başlatılabilir.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/pqa_data
!sed -i 's|^data_dir: .*|data_dir: /content/drive/MyDrive/pqa_data|' config.yaml
!grep data_dir config.yaml

## 5. Tam çalıştırma

`--limit` verme; pasaj/soru sayısını `config.yaml` → `source.hf_dataset.n_articles` ile ayarla
(pilot hedefi ~1k soru ≈ 600-1000 pasaj → `n_articles` ~250-350 civarı, `passages.max_passages_per_article`
değerine bağlı olarak değişir — küçük bir aralıkla deneyip oranı gözlemlemen daha güvenli).

Bağlantı koparsa: bu hücreyi (veya tek tek aşama komutlarını) tekrar çalıştırman yeterli,
tamamlanmış aşamalar `data_dir` altındaki JSONL'lerden anlaşılır ve `vqa` her aşamayı
girdisi hazır olduğu sürece bağımsız çalıştırır.

In [ ]:
!python -m vqa all

## 6. Hugging Face'e yayınla

`config.yaml` → `publish.repo_id` alanını kendi kullanıcı adınla güncelle, sonra giriş yap.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!python -m vqa publish